In [197]:
import warnings
warnings.filterwarnings('ignore') 

In [198]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
import scipy.cluster.hierarchy as sch
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [199]:
df = pd.read_csv('dublinbike-historical-data-2022-11.csv')

In [200]:
df.head()

,STATION ID,TIME,LAST UPDATED,NAME,BIKE_STANDS,AVAILABLE_BIKE_STANDS,AVAILABLE_BIKES,STATUS,ADDRESS,LATITUDE,LONGITUDE
0,1,2022-11-01 00:00:03,2022-10-31 23:51:25,CLARENDON ROW,31,9,22,OPEN,Clarendon Row,53.3409,-6.26250
1,2,2022-11-01 00:00:03,2022-10-31 23:53:48,BLESSINGTON STREET,20,15,5,OPEN,Blessington Street,53.3568,-6.26814
2,3,2022-11-01 00:00:03,2022-10-31 23:50:25,BOLTON STREET,20,17,3,OPEN,Bolton Street,53.3512,-6.26986
3,4,2022-11-01 00:00:03,2022-10-31 23:58:06,GREEK STREET,20,0,20,OPEN,Greek Street,53.3469,-6.27298
4,5,2022-11-01 00:00:03,2022-10-31 23:50:27,CHARLEMONT PLACE,40,31,9,OPEN,Charlemont Street,53.3307,-6.26018


In [201]:
df.tail()

,STATION ID,TIME,LAST UPDATED,NAME,BIKE_STANDS,AVAILABLE_BIKE_STANDS,AVAILABLE_BIKES,STATUS,ADDRESS,LATITUDE,LONGITUDE
162602,113,2022-11-30 23:30:02,2022-11-30 23:26:00,MERRION SQUARE SOUTH,40,15,25,OPEN,Merrion Square South,53.3386,-6.24861
162603,114,2022-11-30 23:30:02,2022-11-30 23:26:47,WILTON TERRACE (PARK),40,40,0,OPEN,Wilton Terrace (Park),53.3337,-6.24834
162604,115,2022-11-30 23:30:02,2022-11-30 23:23:32,KILLARNEY STREET,30,5,25,OPEN,Killarney Street,53.3548,-6.24758
162605,116,2022-11-30 23:30:02,2022-11-30 23:25:26,BROADSTONE,30,28,2,OPEN,Broadstone,53.3547,-6.27231
162606,117,2022-11-30 23:30:02,2022-11-30 23:22:24,HANOVER QUAY EAST,40,30,10,OPEN,Hanover Quay East,53.3437,-6.23175


In [202]:
df.tail()

,STATION ID,TIME,LAST UPDATED,NAME,BIKE_STANDS,AVAILABLE_BIKE_STANDS,AVAILABLE_BIKES,STATUS,ADDRESS,LATITUDE,LONGITUDE
162602,113,2022-11-30 23:30:02,2022-11-30 23:26:00,MERRION SQUARE SOUTH,40,15,25,OPEN,Merrion Square South,53.3386,-6.24861
162603,114,2022-11-30 23:30:02,2022-11-30 23:26:47,WILTON TERRACE (PARK),40,40,0,OPEN,Wilton Terrace (Park),53.3337,-6.24834
162604,115,2022-11-30 23:30:02,2022-11-30 23:23:32,KILLARNEY STREET,30,5,25,OPEN,Killarney Street,53.3548,-6.24758
162605,116,2022-11-30 23:30:02,2022-11-30 23:25:26,BROADSTONE,30,28,2,OPEN,Broadstone,53.3547,-6.27231
162606,117,2022-11-30 23:30:02,2022-11-30 23:22:24,HANOVER QUAY EAST,40,30,10,OPEN,Hanover Quay East,53.3437,-6.23175


In [203]:
print("Dataset shape:", df.shape)

Dataset shape: (162607, 11)


In [204]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 162607 entries, 0 to 162606
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   STATION ID             162607 non-null  int64  
 1   TIME                   162607 non-null  str    
 2   LAST UPDATED           162607 non-null  str    
 3   NAME                   162607 non-null  str    
 4   BIKE_STANDS            162607 non-null  int64  
 5   AVAILABLE_BIKE_STANDS  162607 non-null  int64  
 6   AVAILABLE_BIKES        162607 non-null  int64  
 7   STATUS                 162607 non-null  str    
 8   ADDRESS                162607 non-null  str    
 9   LATITUDE               162607 non-null  float64
 10  LONGITUDE              162607 non-null  float64
dtypes: float64(2), int64(4), str(5)
memory usage: 25.3 MB


In [205]:
df.describe()

,STATION ID,BIKE_STANDS,AVAILABLE_BIKE_STANDS,AVAILABLE_BIKES,LATITUDE,LONGITUDE
count,162607.000000,162607.000000,162607.000000,162607.000000,162607.000000,162607.000000
mean,58.699115,32.000000,19.783503,11.987965,53.345591,-6.264277
std,34.239989,7.488067,11.218428,9.896961,0.007600,0.017951
min,1.000000,16.000000,0.000000,0.000000,53.330100,-6.310020
25%,29.000000,29.000000,11.000000,3.000000,53.339800,-6.275070
50%,58.000000,30.000000,20.000000,10.000000,53.345200,-6.262980
75%,89.000000,40.000000,29.000000,18.000000,53.350900,-6.251560
max,117.000000,40.000000,40.000000,40.000000,53.360000,-6.230850


In [206]:
df.describe()

,STATION ID,BIKE_STANDS,AVAILABLE_BIKE_STANDS,AVAILABLE_BIKES,LATITUDE,LONGITUDE
count,162607.000000,162607.000000,162607.000000,162607.000000,162607.000000,162607.000000
mean,58.699115,32.000000,19.783503,11.987965,53.345591,-6.264277
std,34.239989,7.488067,11.218428,9.896961,0.007600,0.017951
min,1.000000,16.000000,0.000000,0.000000,53.330100,-6.310020
25%,29.000000,29.000000,11.000000,3.000000,53.339800,-6.275070
50%,58.000000,30.000000,20.000000,10.000000,53.345200,-6.262980
75%,89.000000,40.000000,29.000000,18.000000,53.350900,-6.251560
max,117.000000,40.000000,40.000000,40.000000,53.360000,-6.230850


In [207]:
df.nunique()

STATION ID                  113
TIME                       1439
LAST UPDATED             132190
NAME                        113
BIKE_STANDS                  17
AVAILABLE_BIKE_STANDS        41
AVAILABLE_BIKES              41
STATUS                        1
ADDRESS                     113
LATITUDE                     98
LONGITUDE                   113
dtype: int64

In this dataset there is  162,607 rows and 11 columns. Each row is snapshot of one station at one moment in time: the number of stand BIKE_STANDS), the number of bike available AVAILABLE_BIKES) and the number of free stands AVAILABLE_BIKE_STANDS), together with the station name, address, status and coordinates, there are 113 station and 1,439 diferent read times (about every 30 minutes dure the 30 days of November 2022).

## Step 3: Data cleaning
I check miss values, duplicate, constant or redundant columns and the consistency of the read.

In [208]:
# Check the miss values
df.isnull().sum()

STATION ID               0
TIME                     0
LAST UPDATED             0
NAME                     0
BIKE_STANDS              0
AVAILABLE_BIKE_STANDS    0
AVAILABLE_BIKES          0
STATUS                   0
ADDRESS                  0
LATITUDE                 0
LONGITUDE                0
dtype: int64

In [209]:
# Check the duplicate rows
print("Duplicated rows:", df.duplicated().sum())
print("Duplicated station/time readings:", df.duplicated(subset = ['STATION ID', 'TIME']).sum())

Duplicated rows: 0
Duplicated station/time readings: 0


In [210]:
df['STATUS'].value_counts()

STATUS
OPEN    162607
Name: count, dtype: int64

In [211]:
print("Unique names:", df['NAME'].nunique())
print("Unique addresses:", df['ADDRESS'].nunique())

Unique names: 113
Unique addresses: 113


- There are no miss values and no duplicated readings.
- Status have only one value opem), so it is a constant column and give no information to the models.
- Address has the same number of unique values as name (one per station), so it is redundant.
- Last update  is a technical timestamp of the API (when the station last communicated) and is not needed, because we use time as the reading time.
These three columns are removed.

In [212]:

df = df.drop(['STATUS', 'ADDRESS', 'LAST UPDATED'], axis = 1)

df['TIME'] = pd.to_datetime(df['TIME'])

df = df.sort_values(['STATION ID', 'TIME']).reset_index(drop = True)

df.head()

,STATION ID,TIME,NAME,BIKE_STANDS,AVAILABLE_BIKE_STANDS,AVAILABLE_BIKES,LATITUDE,LONGITUDE
0,1,2022-11-01 00:00:03,CLARENDON ROW,31,9,22,53.3409,-6.2625
1,1,2022-11-01 00:30:02,CLARENDON ROW,31,9,22,53.3409,-6.2625
2,1,2022-11-01 01:00:03,CLARENDON ROW,31,9,22,53.3409,-6.2625
3,1,2022-11-01 01:30:03,CLARENDON ROW,31,9,22,53.3409,-6.2625
4,1,2022-11-01 02:00:04,CLARENDON ROW,31,9,22,53.3409,-6.2625
